# PySpark XML Processing

This notebook demonstrates how to read and process XML data in PySpark using the spark-xml library.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

## Download spark-xml Package

In [ ]:
# Download spark-xml JAR file
!wget https://repo1.maven.org/maven2/com/databricks/spark-xml_2.12/0.14.0/spark-xml_2.12-0.14.0.jar -O /content/spark-xml_2.12-0.14.0.jar

# Verify download
!ls -lh /content/spark-xml_2.12-0.14.0.jar

## Create Spark Session with XML Package

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session with spark-xml library
spark = SparkSession.builder \
    .appName('PySpark XML Processing') \
    .config("spark.jars", "/content/spark-xml_2.12-0.14.0.jar") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Create Sample XML Files

In [ ]:
# Create sample XML file - Books
books_xml = '''<?xml version="1.0"?>
<catalog>
    <book id="bk101">
        <author>Gambardella, Matthew</author>
        <title>XML Developer's Guide</title>
        <genre>Computer</genre>
        <price>44.95</price>
        <publish_date>2000-10-01</publish_date>
        <description>An in-depth look at creating applications with XML.</description>
    </book>
    <book id="bk102">
        <author>Ralls, Kim</author>
        <title>Midnight Rain</title>
        <genre>Fantasy</genre>
        <price>5.95</price>
        <publish_date>2000-12-16</publish_date>
        <description>A former architect battles corporate zombies.</description>
    </book>
    <book id="bk103">
        <author>Corets, Eva</author>
        <title>Maeve Ascendant</title>
        <genre>Fantasy</genre>
        <price>5.95</price>
        <publish_date>2000-11-17</publish_date>
        <description>After the collapse of a nanotechnology society.</description>
    </book>
    <book id="bk104">
        <author>Corets, Eva</author>
        <title>Oberon's Legacy</title>
        <genre>Fantasy</genre>
        <price>5.95</price>
        <publish_date>2001-03-10</publish_date>
        <description>In post-apocalypse England.</description>
    </book>
    <book id="bk105">
        <author>Randall, Cynthia</author>
        <title>Lover Birds</title>
        <genre>Romance</genre>
        <price>4.95</price>
        <publish_date>2000-09-02</publish_date>
        <description>When Carla meets Paul at an ornithology conference.</description>
    </book>
</catalog>
'''

# Save to file
with open('/content/books.xml', 'w') as f:
    f.write(books_xml)

print("Created books.xml")

In [ ]:
# Create sample XML file - Employees
employees_xml = '''<?xml version="1.0"?>
<company>
    <employee id="001">
        <name>John Doe</name>
        <department>IT</department>
        <salary>75000</salary>
        <location>New York</location>
    </employee>
    <employee id="002">
        <name>Jane Smith</name>
        <department>HR</department>
        <salary>65000</salary>
        <location>San Francisco</location>
    </employee>
    <employee id="003">
        <name>Bob Johnson</name>
        <department>IT</department>
        <salary>80000</salary>
        <location>New York</location>
    </employee>
    <employee id="004">
        <name>Alice Williams</name>
        <department>Finance</department>
        <salary>90000</salary>
        <location>Chicago</location>
    </employee>
</company>
'''

with open('/content/employees.xml', 'w') as f:
    f.write(employees_xml)

print("Created employees.xml")

## Reading XML Files

### Method 1: Read Simple XML

In [ ]:
# Read books XML file
# rowTag specifies which XML element becomes a row
df_books = spark.read \
    .format("xml") \
    .option("rowTag", "book") \
    .load("/content/books.xml")

print("Books DataFrame:")
df_books.printSchema()
df_books.show(truncate=False)

In [ ]:
# Count records
print(f"Total books: {df_books.count()}")

# Show specific columns
df_books.select("_id", "title", "author", "price", "genre").show(truncate=False)

### Method 2: Read with Attributes

In [ ]:
# Read employees XML with attributes
df_employees = spark.read \
    .format("xml") \
    .option("rowTag", "employee") \
    .option("attributePrefix", "_") \
    .load("/content/employees.xml")

print("Employees DataFrame:")
df_employees.printSchema()
df_employees.show(truncate=False)

## Data Analysis on XML

In [ ]:
from pyspark.sql.functions import col, avg, sum as spark_sum, count, round as spark_round

# Books analysis
print("Books by Genre:")
df_books.groupBy("genre") \
    .agg(
        count("*").alias("count"),
        spark_round(avg("price"), 2).alias("avg_price"),
        spark_round(spark_sum("price"), 2).alias("total_value")
    ) \
    .orderBy(col("count").desc()) \
    .show()

In [ ]:
# Most expensive books
print("Top 3 Most Expensive Books:")
df_books.select("title", "author", "price", "genre") \
    .orderBy(col("price").desc()) \
    .limit(3) \
    .show(truncate=False)

In [ ]:
# Books by author
print("Books per Author:")
df_books.groupBy("author") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

In [ ]:
# Employee analysis
print("Employees by Department:")
df_employees.groupBy("department") \
    .agg(
        count("*").alias("employee_count"),
        spark_round(avg("salary"), 2).alias("avg_salary"),
        spark_sum("salary").alias("total_salary")
    ) \
    .show()

In [ ]:
# Highest paid employees
print("Top 3 Highest Paid Employees:")
df_employees.select("_id", "name", "department", "salary", "location") \
    .orderBy(col("salary").desc()) \
    .limit(3) \
    .show(truncate=False)

## Filtering XML Data

In [ ]:
# Filter books
fantasy_books = df_books.filter(col("genre") == "Fantasy")
print("Fantasy Books:")
fantasy_books.select("title", "author", "price").show(truncate=False)

# Books published in 2000
books_2000 = df_books.filter(col("publish_date").startswith("2000"))
print("\nBooks Published in 2000:")
books_2000.select("title", "publish_date").show(truncate=False)

In [ ]:
# Filter employees
it_dept = df_employees.filter(col("department") == "IT")
print("IT Department Employees:")
it_dept.select("name", "salary", "location").show(truncate=False)

# High earners (> 70000)
high_earners = df_employees.filter(col("salary") > 70000)
print("\nEmployees with Salary > 70000:")
high_earners.select("name", "department", "salary").show(truncate=False)

## Using Spark SQL with XML

In [ ]:
# Register DataFrames as temp views
df_books.createOrReplaceTempView("books")
df_employees.createOrReplaceTempView("employees")

# SQL query on books
sql_result1 = spark.sql("""
    SELECT 
        genre,
        COUNT(*) as book_count,
        ROUND(AVG(price), 2) as avg_price,
        MIN(price) as min_price,
        MAX(price) as max_price
    FROM books
    GROUP BY genre
    ORDER BY book_count DESC
""")

print("Genre Statistics (SQL):")
sql_result1.show()

In [ ]:
# Complex SQL query
sql_result2 = spark.sql("""
    SELECT 
        department,
        location,
        COUNT(*) as employee_count,
        ROUND(AVG(salary), 2) as avg_salary
    FROM employees
    GROUP BY department, location
    ORDER BY department, location
""")

print("Employee Distribution by Department and Location (SQL):")
sql_result2.show()

## Writing Data to XML

In [ ]:
# Save filtered data back to XML
fantasy_books.write \
    .format("xml") \
    .option("rootTag", "catalog") \
    .option("rowTag", "book") \
    .mode("overwrite") \
    .save("/content/fantasy_books_output")

print("Fantasy books saved to XML")

# List output files
!ls -lh /content/fantasy_books_output/

In [ ]:
# Save employee analysis to XML
dept_analysis = df_employees.groupBy("department") \
    .agg(
        count("*").alias("count"),
        avg("salary").alias("avg_salary")
    )

dept_analysis.write \
    .format("xml") \
    .option("rootTag", "departments") \
    .option("rowTag", "department_info") \
    .mode("overwrite") \
    .save("/content/dept_analysis_output")

print("Department analysis saved to XML")

## Working with Nested XML

In [ ]:
# Create nested XML
nested_xml = '''<?xml version="1.0"?>
<orders>
    <order id="1001">
        <customer>
            <name>John Smith</name>
            <email>john@example.com</email>
        </customer>
        <items>
            <item>
                <product>Laptop</product>
                <quantity>1</quantity>
                <price>1200.00</price>
            </item>
            <item>
                <product>Mouse</product>
                <quantity>2</quantity>
                <price>25.00</price>
            </item>
        </items>
        <total>1250.00</total>
    </order>
    <order id="1002">
        <customer>
            <name>Jane Doe</name>
            <email>jane@example.com</email>
        </customer>
        <items>
            <item>
                <product>Keyboard</product>
                <quantity>1</quantity>
                <price>75.00</price>
            </item>
        </items>
        <total>75.00</total>
    </order>
</orders>
'''

with open('/content/orders.xml', 'w') as f:
    f.write(nested_xml)

print("Created nested XML file")

In [ ]:
# Read nested XML
df_orders = spark.read \
    .format("xml") \
    .option("rowTag", "order") \
    .load("/content/orders.xml")

print("Orders DataFrame (Nested):")
df_orders.printSchema()
df_orders.show(truncate=False)

In [ ]:
# Access nested fields
df_orders.select(
    col("_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email").alias("customer_email"),
    col("total")
).show(truncate=False)

In [ ]:
# Explode items array
from pyspark.sql.functions import explode

df_order_items = df_orders.select(
    col("_id").alias("order_id"),
    col("customer.name").alias("customer_name"),
    explode(col("items.item")).alias("item")
).select(
    "order_id",
    "customer_name",
    col("item.product").alias("product"),
    col("item.quantity").alias("quantity"),
    col("item.price").alias("price")
)

print("Exploded Order Items:")
df_order_items.show(truncate=False)

## XML Reading Options

Key options for reading XML:

- **rowTag**: XML tag that becomes a DataFrame row (required)
- **rootTag**: Root tag for writing XML (default: "rows")
- **attributePrefix**: Prefix for XML attributes (default: "_")
- **valueTag**: Tag for text values (default: "_VALUE")
- **ignoreSurroundingSpaces**: Ignore whitespace (default: false)
- **mode**: Parsing mode (PERMISSIVE, DROPMALFORMED, FAILFAST)
- **charset**: Character encoding (default: UTF-8)
- **samplingRatio**: Fraction of data for schema inference (default: 1.0)

In [ ]:
# Example with multiple options
df_with_options = spark.read \
    .format("xml") \
    .option("rowTag", "book") \
    .option("attributePrefix", "attr_") \
    .option("ignoreSurroundingSpaces", "true") \
    .option("mode", "DROPMALFORMED") \
    .load("/content/books.xml")

print("Books with custom options:")
df_with_options.show(5, truncate=False)

In [ ]:
# Stop Spark Session
spark.stop()